In [1]:
import pandas as pd

df = pd.read_csv("dados_final.csv")

df.head()

,Nome,Nota,Data,Genêros,Material Fonte,Tipo,Número de episódios,Classificação,Posicionamento no ranking,Posicionamento de popularidade,Número de membros,Favoritos,Foto,Sinopse
0,Sousou no Frieren,9.27,"Sep 29, 2023 to Mar 22, 2024","['Adventure', 'Award Winning', 'Drama', 'Fanta...",Manga,TV,28,PG-13 - Teens 13 or older,#1,#104,"1,413,150","88,180",https://myanimelist.net/images/anime/1015/1380...,During their decade-long quest to defeat the D...
1,Steel Ball Run: JoJo no Kimyou na Bouken,9.16,"Mar 19, 2026 to ?","['Action', 'Adventure', 'Mystery', 'Supernatur...",Manga,ONA,Unknown,R - 17+ (violence & profanity),#2,#1440,"191,089","6,074",https://myanimelist.net/images/anime/1448/1541...,"In the American Old West, the world's greatest..."
2,Gintama',9.02,"Apr 4, 2011 to Mar 26, 2012","['Action', 'Comedy', 'Sci-Fi']",Manga,TV,51,PG-13 - Teens 13 or older,#11,#406,"612,318","8,591",https://myanimelist.net/images/anime/4/50361.jpg,"After a one-year hiatus, Shinpachi Shimura ret..."
3,Luo Xiaohei Zhanji 2,8.62,"Jul 18, 2025","['Adventure', 'Drama', 'Fantasy']",Original,Movie,1,G - All Ages,#101,#7419,"9,454",74,https://myanimelist.net/images/anime/1288/1518...,When an attack shatters the fragile peace betw...
4,Made in Abyss,8.62,"Jul 7, 2017 to Sep 29, 2017","['Adventure', 'Drama', 'Fantasy', 'Mystery', '...",Web manga,TV,13,R - 17+ (violence & profanity),#102,#92,"1,546,098","47,752",https://myanimelist.net/images/anime/6/86733.jpg,The Abyss—a gaping chasm stretching down into ...


In [2]:
from nltk.tokenize import wordpunct_tokenize
from unidecode import unidecode


def limpar_texto(texto):
    texto = texto.lower()
    texto = unidecode(texto)
    tokens = wordpunct_tokenize(texto)
    tokens = [token for token in tokens if token.isalnum()]
    return " ".join(tokens)


df["texto_limpo"] = df["Sinopse"].apply(limpar_texto)

df[["Sinopse", "texto_limpo"]].head()

,Sinopse,texto_limpo
0,During their decade-long quest to defeat the D...,during their decade long quest to defeat the d...
1,"In the American Old West, the world's greatest...",in the american old west the world s greatest ...
2,"After a one-year hiatus, Shinpachi Shimura ret...",after a one year hiatus shinpachi shimura retu...
3,When an attack shatters the fragile peace betw...,when an attack shatters the fragile peace betw...
4,The Abyss—a gaping chasm stretching down into ...,the abyss a gaping chasm stretching down into ...


In [3]:
import nltk

from nltk.corpus import stopwords


nltk.download("stopwords", quiet=True)

stopwords_en = stopwords.words("english")
stopwords_en = [unidecode(palavra) for palavra in stopwords_en]
stopwords_en = set(stopwords_en)


def remover_stopwords(texto):
    tokens = texto.split()
    tokens = [token for token in tokens if token not in stopwords_en]
    return tokens


df["tokens_sem_stopwords"] = df["texto_limpo"].apply(remover_stopwords)
df["texto_sem_stopwords"] = df["tokens_sem_stopwords"].str.join(" ")

df[["texto_limpo", "tokens_sem_stopwords", "texto_sem_stopwords"]].head()

,texto_limpo,tokens_sem_stopwords,texto_sem_stopwords
0,during their decade long quest to defeat the d...,"[decade, long, quest, defeat, demon, king, mem...",decade long quest defeat demon king members he...
1,in the american old west the world s greatest ...,"[american, old, west, world, greatest, race, b...",american old west world greatest race begin th...
2,after a one year hiatus shinpachi shimura retu...,"[one, year, hiatus, shinpachi, shimura, return...",one year hiatus shinpachi shimura returns edo ...
3,when an attack shatters the fragile peace betw...,"[attack, shatters, fragile, peace, spirit, wor...",attack shatters fragile peace spirit world hum...
4,the abyss a gaping chasm stretching down into ...,"[abyss, gaping, chasm, stretching, depths, ear...",abyss gaping chasm stretching depths earth fil...


In [4]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
matriz_bow = vectorizer.fit_transform(df["texto_sem_stopwords"])

df_bow = pd.DataFrame(
    matriz_bow.toarray(),
    columns=vectorizer.get_feature_names_out()
)

df_bow.head()

,000,01,02,10,104th,11,12,14,15,150,...,zha,zhao,zheng,zhou,zhuan,zimmerman,zodiac,zoku,zoldyck,zu
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
colunas_com_numeros = [col for col in df_bow.columns if any(char.isdigit() for char in col)]

df_bow = df_bow.drop(columns=colunas_com_numeros)

print(f"{len(colunas_com_numeros)} colunas removidas")
df_bow.head()

35 colunas removidas


,abandon,abandoned,abandoning,abashiri,abducted,abduction,abiko,abilities,ability,able,...,zha,zhao,zheng,zhou,zhuan,zimmerman,zodiac,zoku,zoldyck,zu
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
metadados = df[["Nome", "Nota", "Data", "Genêros", "Material Fonte", "Tipo", "Número de episódios", "Classificação", "Posicionamento no ranking", "Posicionamento de popularidade", "Número de membros", "Favoritos", "Foto", "Sinopse"]].reset_index(drop=True)
bow_com_prefixo = df_bow.add_prefix("bow_").reset_index(drop=True)

In [10]:
df_final = pd.concat([metadados, bow_com_prefixo], axis=1)

df_final.head()

,Nome,Nota,Data,Genêros,Material Fonte,Tipo,Número de episódios,Classificação,Posicionamento no ranking,Posicionamento de popularidade,...,bow_zha,bow_zhao,bow_zheng,bow_zhou,bow_zhuan,bow_zimmerman,bow_zodiac,bow_zoku,bow_zoldyck,bow_zu
0,Sousou no Frieren,9.27,"Sep 29, 2023 to Mar 22, 2024","['Adventure', 'Award Winning', 'Drama', 'Fanta...",Manga,TV,28,PG-13 - Teens 13 or older,#1,#104,...,0,0,0,0,0,0,0,0,0,0
1,Steel Ball Run: JoJo no Kimyou na Bouken,9.16,"Mar 19, 2026 to ?","['Action', 'Adventure', 'Mystery', 'Supernatur...",Manga,ONA,Unknown,R - 17+ (violence & profanity),#2,#1440,...,0,0,0,0,0,0,0,0,0,0
2,Gintama',9.02,"Apr 4, 2011 to Mar 26, 2012","['Action', 'Comedy', 'Sci-Fi']",Manga,TV,51,PG-13 - Teens 13 or older,#11,#406,...,0,0,0,0,0,0,0,0,0,0
3,Luo Xiaohei Zhanji 2,8.62,"Jul 18, 2025","['Adventure', 'Drama', 'Fantasy']",Original,Movie,1,G - All Ages,#101,#7419,...,0,0,0,0,0,0,0,0,0,0
4,Made in Abyss,8.62,"Jul 7, 2017 to Sep 29, 2017","['Adventure', 'Drama', 'Fantasy', 'Mystery', '...",Web manga,TV,13,R - 17+ (violence & profanity),#102,#92,...,0,0,0,0,0,0,0,0,0,0


In [11]:
colunas_bow = [coluna for coluna in df_final.columns if coluna.startswith("bow_")]

frequencia_palavras = df_final[colunas_bow].sum().sort_values(ascending=False)

print(f"Total de palavras diferentes: {len(frequencia_palavras)}")

frequencia_palavras.head(10)

Total de palavras diferentes: 6278


bow_written    207
bow_mal        205
bow_rewrite    204
bow_however    111
bow_one         99
bow_world       93
bow_life        89
bow_new         82
bow_school      69
bow_time        66
dtype: int64

In [12]:
frequencia_palavras.tail(10)

bow_indescribable       1
bow_indifference        1
bow_indiscriminately    1
bow_indispensable       1
bow_inducted            1
bow_indulgent           1
bow_indulging           1
bow_industry            1
bow_inexplicably        1
bow_zu                  1
dtype: int64

In [13]:
documentos_por_palavra = (df_final[colunas_bow] > 0).sum().sort_values(ascending=False)
documentos_por_palavra.index = documentos_por_palavra.index.str.replace("bow_", "", regex=False)

documentos_por_palavra.head(10)

mal        205
rewrite    204
written    204
however    102
one         73
world       70
new         60
life        58
time        57
two         53
dtype: int64

In [20]:
df_final["palavras_unicas"] = (df_final[colunas_bow] > 0).sum(axis=1)

df_final[["Nome", "palavras_unicas"]].sort_values("palavras_unicas", ascending=False).head(10)

,Nome,palavras_unicas
128,Spy x Family,130
62,Samurai Champloo,125
204,Kingdom 6th Season,124
219,Ping Pong the Animation,119
127,Shingeki no Kyojin: Kuinaki Sentaku,113
184,Jujutsu Kaisen 2nd Season,113
77,Chou Kaguya-hime!,109
42,Steins;Gate 0,108
157,Hajime no Ippo,108
121,Kono Oto Tomare! Part 2,107
